# Credit Risk Prediction & Default Analysis
**Role:** Machine Learning Analyst  
**Dataset:** `credit (1).csv`  
**Models:** Logistic Regression, Decision Tree, Random Forest, SVM

## Business objective
Predict `default` (0 = good/non-default, 1 = default) to help a bank identify risky applications before lending. The project brief says recall for the default class, F1-score, and ROC-AUC matter alongside accuracy: a false negative can expose the bank to unpaid principal, while a false positive can mean turning away a potentially good customer.

> **Important:** This is an educational prototype, not a production credit decision system. Results depend on this dataset, split, threshold, and assumptions. Do not use demographic/protected attributes as automated rejection grounds without legal, fairness, and compliance review.

## 1. Load and inspect the dataset
The identifier `observation_id` is excluded from model inputs. The target is `default`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

df = pd.read_csv("credit (1).csv")
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing values"))
print("\nTarget distribution:")
display(df["default"].value_counts().rename(index={0:"Good / non-default",1:"Default"}).to_frame("count"))
print("Default rate:", round(df["default"].mean()*100, 2), "%")

## 2. Exploratory data analysis
Check class balance and loan amount distribution. `amount` is treated as the loan amount for the illustrative exposure calculation below.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["default"].value_counts().sort_index().rename(index={0:"Good (0)",1:"Default (1)"}).plot(kind="bar", ax=axes[0], title="Target class counts")
axes[0].set_xlabel("Class"); axes[0].set_ylabel("Applicants")
axes[1].hist(df["amount"], bins=25, edgecolor="black")
axes[1].set_title("Loan amount distribution"); axes[1].set_xlabel("Amount (dataset units)"); axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

## 3. Preprocessing and train/test split
- 80% train / 20% test, stratified by `default`, fixed random seed 42.
- Numeric missing values: median imputation; categorical missing values: most-frequent imputation.
- Categorical fields: one-hot encoding.
- Numeric fields: standard scaling (important for Logistic Regression and SVM).
- Preprocessing is inside each pipeline to avoid leaking test-set information into training.

In [ ]:
target = "default"
X = df.drop(columns=[target, "observation_id"], errors="ignore")
y = df[target].astype(int)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", len(numeric_features), "| Categorical features:", len(categorical_features))

## 4. Train four classifiers
All predictions below use the default probability threshold of 0.50. The Decision Tree uses `min_samples_leaf=5`; Random Forest uses 400 trees. These are baseline settings, not an exhaustive hyperparameter search.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, min_samples_leaf=5),
    "Random Forest": RandomForestClassifier(n_estimators=400, random_state=42, min_samples_leaf=2, n_jobs=-1),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42)
}

fitted_models = {}
predictions = {}
probabilities = {}
rows = []
confusion_matrices = {}

for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fitted_models[name] = pipe
    predictions[name] = y_pred
    probabilities[name] = y_prob
    confusion_matrices[name] = confusion_matrix(y_test, y_pred, labels=[0, 1])
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (default)": precision_score(y_test, y_pred, zero_division=0),
        "Recall (default)": recall_score(y_test, y_pred, zero_division=0),
        "F1 (default)": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results = pd.DataFrame(rows).set_index("Model").sort_values("F1 (default)", ascending=False)
display(results.style.format("{:.3f}"))


## 5. Confusion matrices
For class `1` (default):
- **False Negative (FN):** actually defaulted, model predicted good — potential bad-loan exposure.
- **False Positive (FP):** actually good, model predicted default — possible lost lending opportunity.
- **True Positive (TP):** default correctly flagged.
- **True Negative (TN):** good loan correctly identified.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, (name, cm) in zip(axes.ravel(), confusion_matrices.items()):
    ConfusionMatrixDisplay(cm, display_labels=["Good (0)", "Default (1)"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

for name in fitted_models:
    print("\n", name)
    print(classification_report(y_test, predictions[name], target_names=["Good / non-default", "Default"], zero_division=0))

## 6. Illustrative bank impact
The CSV has an `amount` column. This section estimates:
1. **Potential principal exposed by false negatives** = sum of `amount` for actual defaults predicted as good.
2. **Potential lost opportunity** = 10% of the `amount` sum for good applicants predicted as default. The 10% is a placeholder margin assumption for classroom discussion, **not a real bank fee or guaranteed loss**.

These amounts are not expected losses or accounting figures. A default does not always mean the full principal is lost; recoveries, collateral, repayments, interest, and provisions are not modelled.

In [ ]:
test_amounts = df.loc[X_test.index, "amount"].to_numpy()
y_true = y_test.to_numpy()
impact_rows = []

for name, y_pred in predictions.items():
    false_negative = (y_true == 1) & (y_pred == 0)
    false_positive = (y_true == 0) & (y_pred == 1)
    fn_principal = test_amounts[false_negative].sum()
    fp_amount = test_amounts[false_positive].sum()
    impact_rows.append({
        "Model": name,
        "Missed defaults (FN)": int(false_negative.sum()),
        "Potential principal exposed (FN amount sum)": int(fn_principal),
        "Good applicants flagged (FP)": int(false_positive.sum()),
        "Amount of good loans flagged (FP amount sum)": int(fp_amount),
        "Illustrative opportunity cost at 10%": round(fp_amount * 0.10, 2)
    })

impact = pd.DataFrame(impact_rows).set_index("Model")
display(impact)
print("Note: monetary values are in the dataset's amount units; currency is not specified in the CSV.")

## 7. Recommendation and limitations
Use the metrics together, not accuracy alone. In this fixed holdout run, select the model that best matches the bank's stated priority: catching defaults (recall) while maintaining a useful F1 score. ROC-AUC measures ranking ability across thresholds, so it can favour a different model than the 0.50-threshold metrics.

Before real use:
- Tune the probability threshold using validation data and explicit cost assumptions.
- Perform cross-validation and hyperparameter search; report confidence/variability.
- Assess fairness, explainability, privacy, and applicable lending regulations.
- Validate on newer, representative data and monitor drift.
- Treat predictions as decision support, not automatic approval/rejection.
